In [10]:
import json
from tqdm.auto import tqdm
import os
import dotenv
dotenv.load_dotenv()


True

In [11]:
with open('../Data/documents-with-ids.json', 'rt') as f_in:
    documents = json.load(f_in)

In [3]:
documents[3]

{'intent': 'receive_payment',
 'question': 'i got to receive fucking compensations can i get some ehlp',
 'response': 'To process your compensation payment for the insurance claim, please adhere to the following steps:\n\n1. Access your account by visiting {{WEBSITE_URL}}.\n2. Locate the {{CLAIM_SECTION}} for your specific claim.\n3. Identify the claim that you wish to be compensated for.\n4. Opt for the {{RECEIVE_PAYMENT_OPTION}} that best suits your preference.\n5. Follow the provided instructions to finalize your preferred method of payment and validate the information.\n\nShould you face any complications during this process, do not hesitate to reach out to our customer support team for immediate assistance.',
 'category': 'CLAIMS',
 'id': 'da0592e6'}

In [4]:
prompt_template = """
You are emulating a real customer contacting an insurance company's support.
Based on the following FAQ record, generate 5 natural questions the customer
might ask to get the same information. 

Guidelines:
- Each question should be clear, concise, and sound like something a real customer would say.  
- Do not reuse exact wording from the FAQ record — rephrase naturally.  
- Ensure all questions are relevant to the intent.  
- Cover variations (short, direct, polite, urgent, explanatory).  
- Keep questions simple and customer-friendly.  

FAQ record:
intent: {intent}
question: {question}
response: {response}

Return only valid JSON in this format (no explanations, no code blocks):

{{
  "questions": [
    "question1",
    "question2",
    "question3",
    "question4",
    "question5"
  ]
}}
""".strip()

In [5]:
from openai import OpenAI

client = OpenAI(
    base_url="https://fuzzy-space-couscous-v56xpw9q49g269gv-11434.app.github.dev/v1",
    api_key='ollama',
)

def generate_questions(doc):
    prompt = prompt_template.format(**doc)

    response = client.chat.completions.create(
        model='phi3',
        messages=[{"role": "user", "content": prompt}]
    )
    json_response = response.choices[0].message.content
    return json_response



# # IMPORTANT: set base_url to your codespaces endpoint
# client = OpenAI(
#     base_url="https://fuzzy-space-couscous-v56xpw9q49g269gv-11434.app.github.dev/v1",
#     api_key="ollama"  # dummy, required by OpenAI client
# )

# def generate_questions(doc):
#     prompt = prompt_template.format(**doc)

#     response = client.chat.completions.create(
#         model="phi3",  # must match the ollama model you pulled
#         messages=[{"role": "user", "content": prompt}]
#     )
#     return response.choices[0].message.content



In [ ]:
results = {}

for doc in tqdm(documents): 
    doc_id = doc['id']
    if doc_id in results:
        continue

    questions = generate_questions(doc)
    results[doc_id] = questions
    with open('results.json', 'w') as json_file:
        json.dump(results, json_file)

In [6]:
# documents: işlenecek veri listesi
batch_size = 40
results_dir = "results_batches"
os.makedirs(results_dir, exist_ok=True)

batch_results = {}
batch_index = 0

for i, doc in enumerate(tqdm(documents, desc="Processing documents")):
    doc_id = doc['id']
    if doc_id in batch_results:
        continue

    # Soruları üret
    try:
        questions = generate_questions(doc)
    except Exception as e:
        print(f"Error for doc {doc_id}: {e}")
        questions = None

    batch_results[doc_id] = questions

    # batch_size kadar işlendiyse kaydet ve sıfırla
    if (i + 1) % batch_size == 0 or (i + 1) == len(documents):
        batch_file = os.path.join(results_dir, f"results_batch_{batch_index}.json")
        with open(batch_file, "w", encoding="utf-8") as f:
            json.dump(batch_results, f, indent=2, ensure_ascii=False)
        print(f"✅ Batch {batch_index} saved, processed {i + 1} documents")
        
        batch_results = {}
        batch_index += 1

# Son olarak tüm batch dosyalarını birleştir
all_results = {}
for fname in sorted(os.listdir(results_dir)):
    if fname.endswith(".json"):
        with open(os.path.join(results_dir, fname), "r", encoding="utf-8") as f:
            data = json.load(f)
            all_results.update(data)

# Tüm veriyi tek bir dosyada kaydet
with open("results_all.json", "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=2, ensure_ascii=False)

print("✅ All batches merged into results_all.json")


Processing documents:   0%|          | 0/390 [00:00<?, ?it/s]

Error for doc 5068cd34: Error code: 400
Error for doc 5782e15e: Error code: 400
Error for doc 58cdb525: Error code: 400
Error for doc d23a3899: Error code: 400
✅ Batch 0 saved, processed 40 documents
✅ Batch 1 saved, processed 80 documents
Error for doc b758aa4e: Error code: 400
Error for doc 8bcfbb17: Error code: 400
✅ Batch 2 saved, processed 120 documents
Error for doc 3e85558b: Error code: 400
✅ Batch 3 saved, processed 160 documents
Error for doc 2bc68926: Error code: 400
Error for doc c97608b5: Error code: 400
✅ Batch 4 saved, processed 200 documents
Error for doc 752ea67d: Error code: 502
Error for doc 88c0fe02: Error code: 502
Error for doc e42aba1b: Error code: 502
Error for doc 1ac0028b: Error code: 502
Error for doc 4c793d2d: Error code: 502
Error for doc de67c519: Error code: 502
Error for doc cca33311: Error code: 502
Error for doc 87d36bdb: Error code: 502
Error for doc f1a97444: Error code: 502
Error for doc e00676ef: Error code: 502
Error for doc 5e71c3af: Error code: 5

In [4]:
import json
import re

def load_results(filename="results_all.json"):
    """JSON dosyasını yükler."""
    with open(filename, "rt") as f_in:
        return json.load(f_in)

def clean_json_string(json_string):
    """Model çıktısını temizler, geçerli JSON kısmını döndürür."""
    if not json_string:  # None veya boş string ise
        return None
    # Kod bloklarını kaldır
    cleaned = re.sub(r'```json|```', '', str(json_string)).strip()
    # JSON yapısını yakala (liste veya obje)
    match = re.search(r'(\[.*\]|\{.*\})', cleaned, re.DOTALL)
    return match.group(0) if match else cleaned

def parse_results(results: dict):
    """
    Temizlenmiş JSON sonuçlarını döndürür.
    None veya boş olan id'leri separate listede tutar.
    """
    parsed_results = {}
    null_ids = []

    for doc_id, json_questions in results.items():
        cleaned_json = clean_json_string(json_questions)
        if cleaned_json is None:
            null_ids.append(doc_id)
            continue
        try:
            parsed_results[doc_id] = json.loads(cleaned_json)
        except json.JSONDecodeError:
            null_ids.append(doc_id)

    return parsed_results, null_ids


In [5]:
results = load_results("results_all.json")
parsed_results, null_ids = parse_results(results)

print("Boş/Geçersiz id sayısı:", len(null_ids))

Boş/Geçersiz id sayısı: 224


In [8]:
parsed_results

{'da0592e6': {'questions': ['How can I receive compensation for my insurance claim?',
   'What steps do I need to follow to get paid after filing a claim, please guide me quickly.',
   'Is there an efficient way to submit and confirm the payment process post-claim submission?',
   "I'm facing difficulties with sorting out payments for my insurance payout; can someone assist immediately?",
   'After settling claims at your firm, what is involved in receiving a related reimbursement or compensation?']},
 'c529ffe0': {'questions': ['Where can I receive the insurance payment due to my claim?',
   'How do I get paid for a valid insurance claim?',
   "I'm trying to finalize my insurance payout. Could you guide me through it on your website, please?",
   'Could someone show me how exactly to receive payments from an issued claim step by step using the interface? I need help.',
   'Is there a section where customers can verify and complete their payment option for claims?']},
 '0a7a131b': {'qu

In [14]:
doc_index = {d['id']: d for d in documents}

In [15]:
final_results = []

# for doc_id, questions in parsed_results.items():
#     category = doc_index[doc_id]['category']
#     for q in questions:
#         final_results.append((q, category, doc_id))

for doc_id, data in parsed_results.items():
    # Ensure the doc_id exists in the doc_index to retrieve the category
    if doc_id in doc_index:
        category = doc_index[doc_id]['category']
        
        # Check if `data` is already a list of questions or a dictionary containing 'questions'
        if isinstance(data, list):
            questions = data  # If it's already a list of questions
        elif isinstance(data, dict):
            questions = data.get('questions', [])  # If it's a dict, extract 'questions'
        
        for q in questions:
            final_results.append((q, category, doc_id))

In [16]:
final_results

[('How can I receive compensation for my insurance claim?',
  'CLAIMS',
  'da0592e6'),
 ('What steps do I need to follow to get paid after filing a claim, please guide me quickly.',
  'CLAIMS',
  'da0592e6'),
 ('Is there an efficient way to submit and confirm the payment process post-claim submission?',
  'CLAIMS',
  'da0592e6'),
 ("I'm facing difficulties with sorting out payments for my insurance payout; can someone assist immediately?",
  'CLAIMS',
  'da0592e6'),
 ('After settling claims at your firm, what is involved in receiving a related reimbursement or compensation?',
  'CLAIMS',
  'da0592e6'),
 ('Where can I receive the insurance payment due to my claim?',
  'CLAIMS',
  'c529ffe0'),
 ('How do I get paid for a valid insurance claim?', 'CLAIMS', 'c529ffe0'),
 ("I'm trying to finalize my insurance payout. Could you guide me through it on your website, please?",
  'CLAIMS',
  'c529ffe0'),
 ('Could someone show me how exactly to receive payments from an issued claim step by step us

In [17]:
import pandas as pd
df = pd.DataFrame(final_results, columns=['question', 'category', 'document'])
df.head()

,question,category,document
0,How can I receive compensation for my insuranc...,CLAIMS,da0592e6
1,What steps do I need to follow to get paid aft...,CLAIMS,da0592e6
2,Is there an efficient way to submit and confir...,CLAIMS,da0592e6
3,I'm facing difficulties with sorting out payme...,CLAIMS,da0592e6
4,"After settling claims at your firm, what is in...",CLAIMS,da0592e6


In [18]:
df.shape

(785, 3)

In [19]:
df.isnull().sum()

question    0
category    0
document    0
dtype: int64

In [20]:
df.to_csv('../Data/ground-truth-data.csv', index=False)